In [1]:
import importlib
import hdf_utils
from pathlib import Path
import os
from tqdm.auto import tqdm
import datetime
import sys
import h5py
sys.path.append('/Users/michaellavender/Documents/BUSPC/batch_processing')
from binary_to_hdf import whole_binary_to_hdf

In [2]:
importlib.reload(hdf_utils)

<module 'hdf_utils' from '/Users/michaellavender/Documents/BUSPC/processing/hdf_utils.py'>

# Conversion

In [2]:
from histutils.convert.__main__ import convert_DMC_to_hdf5
dmcfn = Path('/Volumes/Elements/1387/2014-03-30/2014-03-30T10-46-CamSer7196.DMCdata')
outfile = Path('/Volumes/Elements/hdf/2014-03-30T10-46-CamSer7196.h5')
outfile.parent.mkdir(parents=True, exist_ok=True)

In [4]:
# nmea and xml same name as hdf
convert_DMC_to_hdf5(dmcfn, outfile, {"header_bytes": 4})


/Users/michaellavender/Documents/BUSPC/histutils/src/histutils/timedmc.py:72: UserWarning: no explicit representation of timezones available for np.datetime64
  tstart = np.datetime64(tstart)


{'Nframe_extract': 615300,
 'Nmeta': 2,
 'fliplr': False,
 'flipud': False,
 'frame_bytes': 524292,
 'header_bytes': 4,
 'i_rel': array([     0,      1,      2, ..., 615297, 615298, 615299],
      shape=(615300,)),
 'image_bytes': 524288,
 'image_pixels': 262144,
 'kinetic_sec': 0.018867479540576874,
 'rotccw': 0,
 'startUTC': datetime.datetime(2014, 3, 30, 10, 46, 38, tzinfo=datetime.timezone.utc),
 'transpose': False,
 'xy_actual': (512, 512),
 'xy_bin': (1, 1),
 'xy_pixel': (512, 512)}
first raw frame index: 1, last raw frame index: 615300
raw frames cover 2014-03-30T10:46:38.000000000 to 2014-03-30T14:00:07.141293800
writing /Volumes/Elements/hdf/2014-03-30T10-46-CamSer7196.h5 from 2014-03-30T10:46:38.000000000 to 2014-03-30T14:00:07.141293800er7196.h5
writing frame 615290 of 615300 to /Volumes/Elements/hdf/2014-03-30T10-46-CamSer7196.h5

In [ ]:
# nmea and xml different name as hdf
convert_DMC_to_hdf5(dmcfn, outfile, {"header_bytes": 4, 'nmeaFile': '/Volumes/4TB #1 HST1/2014-04-17ext/2014-04-17T07-00-CamSer7196.nmea', 'xmlFile': '/Volumes/4TB #1 HST1/2014-04-17ext/2014-04-17T07-00-CamSer7196.xml'})

In [ ]:
# nmea and xml same name as hdf - no time string
start_time = hdf_utils.parse_log_for_start_time(log_fn)
convert_DMC_to_hdf5(dmcfn, outfile, {"header_bytes": 4, "startUTC": startUTC})

# Rough Keogram

In [4]:
hdf_fn = Path('/Volumes/Elements/hdf/2013-03-30/2014-03-30T10-46-CamSer7196.h5')
out_dir = Path('/Volumes/Elements/hdf/2013-03-30/')
hdf_utils.make_keogram_rougher(hdf_path=hdf_fn, out_dir=out_dir, bin_size_seconds=60, sample_interval=3600)

computing norm:   0%|          | 0/4 [00:00<?, ?chunk/s]

bins:   0%|          | 0/170 [00:00<?, ?bin/s]

keogram
extra buffer
saving


# Batch rough keograms

In [6]:
import os
hdf_parent_folder = '/Volumes/TOSHIBA EXT/HDF'
skip_dirs = {"HDF Tests", "4tb_drive1"}
files = []

# Get files
for root, dirs, filenames in os.walk(hdf_parent_folder):
    dirs[:] = [d for d in dirs if d not in skip_dirs]
    for name in filenames:
        if name.endswith(".h5"):
            files.append(os.path.join(root, name))

files = [Path(f) for f in files]
files = [f for f in files if not f.with_suffix(".png").exists()]

In [10]:
# Process
done = []
errors = []
for file in tqdm(files):
    print(file.name)
    out_dir = file.parent
    try:
        hdf_utils.make_keogram_rougher(hdf_path=file, out_dir=out_dir, bin_size_seconds=60, sample_interval=3600)
        done.append(file)
    except Exception as e:
        errors.append((file, e))
        print(f"  failed: {e}")

  0%|          | 0/6 [00:00<?, ?it/s]

2013-03-24T05-55-CamSer7196.h5
  failed: frames per bin 1344120 must be less than sample interval 3600
2013-04-20-CamSer7196.h5


computing norm:   0%|          | 0/5 [00:00<?, ?chunk/s]

bins:   0%|          | 0/211 [00:00<?, ?bin/s]

keogram
extra buffer
saving
2013-04-20-CamSer7196_reduced_time.h5
  failed: frames per bin 3710 must be less than sample interval 3600
2013-04-23T07-00-CamSer1387.h5


...

bins:   0%|          | 0/211 [00:00<?, ?bin/s]

keogram
extra buffer
saving
2013-04-28T07-00-CamSer1387.h5


computing norm:   0%|          | 0/5 [00:00<?, ?chunk/s]

bins:   0%|          | 0/119 [00:00<?, ?bin/s]

keogram
extra buffer
saving


Errors: 

[(PosixPath('/Volumes/TOSHIBA EXT/HDF/2013-03-24/2013-03-24T05-55-CamSer7196.h5'),
  ValueError('frames per bin 1344120 must be less than sample interval 3600')),
 (PosixPath('/Volumes/TOSHIBA EXT/HDF/2013-04-20/7196/2013-04-20-CamSer7196_reduced_time.h5'),
  ValueError('frames per bin 3710 must be less than sample interval 3600'))]


# Hourly Videos & Keograms

In [5]:
import keogram_consumer_hourly as kch
import keogram_consumer
import stats_consumer
import video_consumer
importlib.reload(kch)
importlib.reload(keogram_consumer)
importlib.reload(stats_consumer)
importlib.reload(video_consumer)
importlib.reload(hdf_utils)

<module 'hdf_utils' from '/Users/michaellavender/Documents/BUSPC/processing/hdf_utils.py'>

In [6]:
# Setup
start = datetime.datetime(2013, 4, 14, 7, 40, 0, tzinfo=datetime.timezone.utc)
start_unix = start.timestamp() # needed for norm

hdf_fn = Path('/Volumes/Elements/hdf/2013-03-30/2014-03-30T10-46-CamSer7196.h5')
out_dir = Path('/Volumes/Elements/hdf/2013-03-30/output')
out_dir.mkdir(exist_ok=True, parents=True)

In [7]:
# Get norm
f = h5py.File(hdf_fn, 'r')
imgs = f['rawimg']
ut = f['ut1_unix']
# start_idx = hdf_utils.find_closest_item(ut, start_unix)
norm = hdf_utils.compute_norm(imgs, ut, 1, low_percentile=0.1, high_percentile=99.9)

computing norm:   0%|          | 0/233 [00:00<?, ?chunk/s]

In [ ]:
norm = hdf_utils.make_hourly_videos_keograms(hdf_path=hdf_fn, out_dir=out_dir, bin_size=5, playback_speed=15, output_hz=1, norm=norm)

videos:   0%|          | 0/5 [00:00<?, ?video/s]

CamSer7196_20140330_0:   0%|          | 0/42507 [00:00<?, ?frame/s]

finalzing video


CamSer7196_20140330_1:   0%|          | 0/190805 [00:00<?, ?frame/s]

finalzing video


CamSer7196_20140330_2:   0%|          | 0/190804 [00:00<?, ?frame/s]

finalzing video


CamSer7196_20140330_3:   0%|          | 0/190805 [00:00<?, ?frame/s]

# Helper Funcitions

In [ ]:
import datetime
import h5py
def get_time_range(hdf_fn):
    with h5py.File(hdf_fn, 'r') as f:
        ut = f['ut1_unix']
        start_epoch, end_epoch = ut[0], ut[-1]
        start = datetime.datetime.fromtimestamp(start_epoch, tz=datetime.timezone.utc)
        end = datetime.datetime.fromtimestamp(end_epoch, tz=datetime.timezone.utc)

    print(start.strftime("%Y-%m-%d %H:%M:%S"), end.strftime("%Y-%m-%d %H:%M:%S"))
    return start, end